In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import os
import seaborn as sns
import pandas as pd
from collections import Counter
from sklearn.cluster import OPTICS
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA


## Load datasets

In [ ]:
path = '/content/drive/MyDrive/cold_heads_monitoring/Dataset'
arrays = []
data_list = []
lens = []

for filename in os.listdir(path):
    if filename.endswith('.pkl'):
        with open(os.path.join(path, filename), 'rb') as f:

            arr = np.asarray(pickle.load(f))[:, 1:2, :]
            lens.append(len(arr))
            arrays.append(arr)
            data_list.append(os.path.splitext(filename)[0])



In [ ]:
all_data = arrays[2]


## Balance datasets

In [ ]:
min_len = 2000

balanced_arrays = [arr[:min_len] for arr in arrays]

data = np.concatenate(balanced_arrays, axis=0)

data = np.squeeze(data,1)
data.shape


In [ ]:
def calculate_features(window):

    mean = np.mean(window)

    std_dev = np.std(window)

    max_val = np.max(window)

    min_val = np.min(window)

    peak_to_peak = max_val - min_val

    return [mean, std_dev, max_val, min_val, peak_to_peak]


In [ ]:
def normalize_data(data):

    mean = np.mean(data, axis=0)
    std_dev = np.std(data, axis=0)

    normalized_data = (data - mean) / std_dev

    return normalized_data


In [ ]:
pca = PCA(n_components=2)

def preprocessing(data):

    # Historical subwindows use the step stored in overlap.
    window_size = 30
    overlap = 2

    windows = []
    for signal in data:
        n = len(signal)
        indices = np.arange(0, n - window_size + 1, overlap)
        windows.extend(signal[i:i+window_size] for i in indices)

    features = np.array([calculate_features(window) for window in windows])

    # Historical shuffle has no fixed seed.
    np.random.shuffle(features)

    return features


## Training DBSCAN

In [ ]:
dataset = preprocessing(data)[:1000]

optics = OPTICS(min_samples=10, xi=0.79, min_cluster_size=0.05)
optics.fit(dataset)
clustering_labels = optics.labels_


In [ ]:
n_clusters = len(set(clustering_labels)) - (1 if -1 in clustering_labels else 0)
print("Number of clusters:", n_clusters)

n_anomalies = list(clustering_labels).count(-1)
print("Number of noise points:", n_anomalies)


In [ ]:
df = pd.DataFrame({'X': dataset[:, 0], 'Y': dataset[:, 1], 'Cluster': clustering_labels})

sns.scatterplot(data=df, x='X', y='Y', hue='Cluster', palette='viridis', alpha=0.5)
plt.show()


## Inference

In [ ]:
def distance(nuovo_campione, data, clustering_labels, metric='chebyshev'):

    cluster_centroids = []
    for label in np.unique(clustering_labels):
        cluster_points = data[clustering_labels == label]
        centroid = np.mean(cluster_points, axis=0)
        cluster_centroids.append(centroid)

    cluster_centroids = np.array(cluster_centroids)

    distanze = pairwise_distances(nuovo_campione.reshape(1, -1), cluster_centroids, metric=metric)

    cluster_più_vicino = np.argmin(distanze)
    return cluster_più_vicino


In [ ]:
min_len = 2000

inference_data = all_data[:min_len]

inference_data = np.squeeze(inference_data,1)
inference_data.shape


In [ ]:
inference_data = preprocessing(inference_data)


In [ ]:
df2 = pd.DataFrame({'X': inference_data[:, 0], 'Y': inference_data[:, 1]})

sns.scatterplot(data=df2, x='X', y='Y', palette='viridis', alpha=0.5)
sns.scatterplot(data=df, x='X', y='Y', hue='Cluster', palette='viridis', alpha=0.5)
plt.show()


In [ ]:
result = []
for elem in inference_data:
  result.append(distance(elem, dataset, clustering_labels))
count = Counter(result)
elemento_piu_ripetuto = count.most_common(1)[0][0]
print(f"Nearest cluster for the new sample: {elemento_piu_ripetuto}")


In [ ]:
with open('dbscan_model.pkl', 'wb') as model_file:
    pickle.dump(optics, model_file)


In [ ]:
new_labels = optics.fit_predict(data)
